# Baomai R42 — review PR127 annotations

**Use a standard CPU runtime. Run all cells. No GPU, training, model inference, API key, or account token is needed.**

This notebook retrieves the immutable R42 review capsule, verifies its identity and row operations, opens a review of the original pixels, and downloads the capsule in Colab. It does not change the model or active dataset.

- All 12 selected images and 507 starting rows were reviewed.
- 166 existing rows were revised; one unsupported row was removed and one genuinely missing front was added. Total 507 rows remains unchanged.
- Complete assistant-reviewed candidates: **frame_0146, frame_0371, frame_0383** (129 fronts).
- Nine whole images remain held, with 55 source rows carrying explicit unresolved fields. Their useful provisional edits are preserved separately.
- The candidate images have **no assigned training split** and are not an independent test set. Assistant review is not human sign-off.

The self-contained `review.html` works offline after extracting the ZIP. Choose an image and a source row to see original pixels, the working baseline, the proposal, and the exact reason. No coordinate comes from a model prediction.


In [ ]:
from pathlib import Path
import base64, hashlib, html, json, subprocess, sys, tempfile, urllib.request, zipfile

# Immutable Git object and content hashes. A branch moving cannot substitute a different capsule.
REPOSITORY = "Siuuuers/dl"
ARCHIVE_BLOB = "f9488f0c17062bb87b901620f1ace714995a9543"
ARCHIVE_SHA256 = "716f81f9b72fffccfb52120d90b30a50b3d605ebd649736ed1f5ad189b4cb2ef"
ARCHIVE_BYTES = 4181563
MANIFEST_SHA256 = "5fad09d5e1922a4b176200459bcd53a698577472feb05a03d1be329aa45896b7"
LOCAL_REVIEW_ARCHIVE = ""  # Optional: an already downloaded R42 review.zip, for offline use.
DOWNLOAD_BUNDLE = True

work = Path(tempfile.mkdtemp(prefix="baomai_r42_"))
archive_path = work / "baomai_front_review_r42.zip"
if LOCAL_REVIEW_ARCHIVE:
    content = Path(LOCAL_REVIEW_ARCHIVE).read_bytes()
else:
    url = f"https://api.github.com/repos/{REPOSITORY}/git/blobs/{ARCHIVE_BLOB}"
    request = urllib.request.Request(url, headers={"Accept":"application/vnd.github+json", "User-Agent":"baomai-r42-review"})
    with urllib.request.urlopen(request, timeout=90) as response:
        result = json.load(response)
    assert result["sha"] == ARCHIVE_BLOB and result["encoding"] == "base64", "Unexpected Git object"
    content = base64.b64decode(result["content"])
assert len(content) == ARCHIVE_BYTES and hashlib.sha256(content).hexdigest() == ARCHIVE_SHA256, "Capsule identity mismatch"
archive_path.write_bytes(content)
folder = work / "review"
with zipfile.ZipFile(archive_path) as z:
    items = z.infolist()
    names = [x.filename for x in items]
    assert len(names) == len(set(names)), "Duplicate ZIP member"
    assert sum(x.file_size for x in items) < 100_000_000, "Unexpected expanded size"
    assert z.testzip() is None, "CRC failure"
    inventory = json.loads(z.read("inventory.json"))
    assert set(names) == set(inventory) | {"inventory.json"}, "Inventory mismatch"
    for item in items:
        rel = Path(item.filename)
        assert not rel.is_absolute() and ".." not in rel.parts and "\\" not in item.filename
        assert not item.is_dir() and ((item.external_attr >> 16) & 0o170000) != 0o120000
        payload = z.read(item.filename)
        if item.filename in inventory:
            assert inventory[item.filename] == {"sha256":hashlib.sha256(payload).hexdigest(), "bytes":len(payload)}, item.filename
        target = folder / rel
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(payload)
assert hashlib.sha256((folder / "manifest.json").read_bytes()).hexdigest() == MANIFEST_SHA256
checked = subprocess.run([sys.executable, str(folder / "review.py"), "--check-capsule", str(archive_path)], check=True, capture_output=True, text=True)
verification = json.loads(checked.stdout)
print("Verified all 12 images and 507 baseline rows.")
print("Complete annotation candidates:", ", ".join(verification["annotation_complete_ids"]))
print("Held whole images:", ", ".join(verification["held_image_ids"]))
print("No training or model promotion occurred.")
print("Capsule:", archive_path)


In [ ]:
from IPython.display import HTML, display
viewer = (folder / "review.html").read_text()
display(HTML('<iframe title="R42 original-pixel review" sandbox="allow-scripts" style="width:100%;height:1100px;border:0" srcdoc="' + html.escape(viewer, quote=True) + '"></iframe>'))
if DOWNLOAD_BUNDLE:
    try:
        from google.colab import files
    except ImportError:
        print("Open the capsule path printed above; review.html is self-contained after extraction.")
    else:
        files.download(str(archive_path))


## What the comparison means

On only the three complete images, the same 218 saved predictions are matched to 129 references. The working labels produce 121 matches, 8 unmatched references and 97 unmatched predictions. The reviewed labels produce 124, 5 and 94 respectively; 45 of the 94 unmatched predictions satisfy the scorer's duplicate criterion. **These are changes in labels and matching, not a better model.**

The source labels, working pilot labels and reviewed proposals remain distinct in the capsule. All nine held images are excluded from this comparison and from `complete-candidates/labels/`.

The next experiment should change one thing at a time: reviewed training data, with the architecture and inference settings fixed. It first needs an explicit training assignment and a leakage-aware comparison plan. Do not use the 320 mined pool as a new independent test set, or delete real partly visible fronts simply to lower an unmatched count.
